# 02_Preprocessing.ipynb: Causal Time-Series Protocol & Feature Engineering (Version 2 Rebuild)

Tuân thủ nghiêm ngặt **AGENTS.md** và **PROCESS.md**:
- Bóc tách thời gian rõ ràng, giữ timestamp naive, không rò rỉ thông tin tương lai.
- Mọi biến đổi và lựa chọn đặc trưng hoàn toàn causal, không fit tham số toàn cục.
- Cung cấp các API cốt lõi cho bài toán Primary (Cửa sổ rủi ro chất lượng) và Secondary (QC inspection triage).
- Bộ kiểm tra tự động (Small synthetic checks) chạy nhanh xác thực tính bất biến nhân quả, xử lý ties và an toàn chỉ số.

In [1]:
import os
import sys
import hashlib
import numpy as np
import pandas as pd
from pathlib import Path

# 1. Bootstrap PROJECT_ROOT and VERSION_DIR
def find_project_root(start_path=None):
    p = Path(start_path).resolve() if start_path else Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / "AGENTS.md").exists() and (parent / "data" / "secom").exists():
            return parent
    raise RuntimeError(f"Could not locate project root from {p}")

PROJECT_ROOT = find_project_root()
VERSION_DIR = PROJECT_ROOT / "Version 2"

DATA_SHA256 = "20f0e7ee434f7dcbae0eea9ffff009a2b57f42d6b0dc9a5bd4f00782c0a3374c"
LABELS_SHA256 = "126884cf453705c9e61a903fe906f0665a3b45ce3639e621edc5c93c89627e03"

print(f"[BOOTSTRAP] PROJECT_ROOT: {PROJECT_ROOT}")
print(f"[BOOTSTRAP] VERSION_DIR:  {VERSION_DIR}")


[BOOTSTRAP] PROJECT_ROOT: D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM
[BOOTSTRAP] VERSION_DIR:  D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM\Version 2


In [2]:
# 2. Ingestion & Preprocessing API: load_raw
def load_raw(data_dir=None):
    """
    Load raw SECOM dataset with strict SHA256 integrity checks, quote-aware timestamp parsing,
    and stable chronological sorting preserving original row IDs.
    Returns DataFrame of shape (1567, 593): timestamp, raw_row_id, fail, sensor_0..sensor_589.
    """
    if data_dir is None:
        data_dir = PROJECT_ROOT / "data" / "secom"
    else:
        data_dir = Path(data_dir)
    data_file = data_dir / "secom.data"
    labels_file = data_dir / "secom_labels.data"
    
    with open(data_file, "rb") as f:
        actual_d_hash = hashlib.sha256(f.read()).hexdigest()
    assert actual_d_hash == DATA_SHA256, f"Data SHA256 mismatch: {actual_d_hash} vs {DATA_SHA256}"
    
    with open(labels_file, "rb") as f:
        actual_l_hash = hashlib.sha256(f.read()).hexdigest()
    assert actual_l_hash == LABELS_SHA256, f"Labels SHA256 mismatch: {actual_l_hash} vs {LABELS_SHA256}"
    
    labels_records = []
    with open(labels_file, "r", encoding="utf-8") as f:
        for idx, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            parts = line.split(" ", 1)
            raw_label = int(parts[0])
            assert raw_label in {-1, 1}, f"Row {idx} has invalid raw label {raw_label}, expected -1 or 1"
            raw_ts_str = parts[1].replace('"', '').strip()
            fail = 1 if raw_label == 1 else 0
            labels_records.append((idx, fail, raw_ts_str))
            
    lbl_df = pd.DataFrame(labels_records, columns=["raw_row_id", "fail", "raw_ts"])
    lbl_df["timestamp"] = pd.to_datetime(lbl_df["raw_ts"], format="%d/%m/%Y %H:%M:%S")
    
    sensor_df = pd.read_csv(data_file, sep=r"\s+", header=None)
    sensor_df.columns = [f"sensor_{i}" for i in range(590)]
    
    merged = pd.concat([lbl_df[["timestamp", "raw_row_id", "fail"]], sensor_df], axis=1)
    sorted_df = merged.sort_values("timestamp", kind="mergesort").reset_index(drop=True)
    
    assert sorted_df.shape == (1567, 593), f"Expected shape (1567, 593), got {sorted_df.shape}"
    assert sorted_df["fail"].sum() == 104, f"Expected 104 fails, got {sorted_df['fail'].sum()}"
    return sorted_df


In [3]:
# 3. Causal Feature Engineering APIs: feature_row & make_features
def feature_row(raw, origin, delay_h=6, history_h=168, lead_h=6):
    """
    Extract strictly causal feature vector at forecast origin.
    Available raw history: timestamp >= origin - W and timestamp + delay <= origin.
    Includes 24 history/QC/calendar concepts and 6 sensor families per channel (3540 features).
    Total features: 3564.
    Returns (series_float, max_feature_available).
    """
    origin = pd.Timestamp(origin)
    w_start = origin - pd.Timedelta(hours=history_h)
    delay_td = pd.Timedelta(hours=delay_h)
    
    hist_mask = (raw["timestamp"] >= w_start) & ((raw["timestamp"] + delay_td) <= origin)
    hist = raw.loc[hist_mask].copy()
    
    # Stable sort by timestamp and raw_row_id before tail/last
    if "raw_row_id" in hist.columns:
        hist = hist.sort_values(["timestamp", "raw_row_id"], kind="mergesort")
    else:
        hist = hist.sort_values("timestamp", kind="mergesort")
    
    features = {}
    if len(hist) > 0:
        max_feature_available = (hist["timestamp"] + delay_td).max()
    else:
        max_feature_available = pd.NaT
        
    # Sub-windows within history for QC labels
    t24_start = origin - pd.Timedelta(hours=24)
    t72_start = origin - pd.Timedelta(hours=72)
    
    h24 = hist[hist["timestamp"] >= t24_start]
    h72 = hist[hist["timestamp"] >= t72_start]
    h168 = hist
    
    n24 = len(h24)
    n72 = len(h72)
    n168 = len(h168)
    
    features["n_avail24"] = float(n24)
    features["n_avail72"] = float(n72)
    features["n_avail168"] = float(n168)
    
    features["fail_rate24"] = float(h24["fail"].mean()) if n24 > 0 else np.nan
    features["fail_rate72"] = float(h72["fail"].mean()) if n72 > 0 else np.nan
    features["fail_rate168"] = float(h168["fail"].mean()) if n168 > 0 else np.nan
    
    sensor_cols = [c for c in raw.columns if c.startswith("sensor_")]
    tot_cells24 = n24 * len(sensor_cols)
    tot_cells72 = n72 * len(sensor_cols)
    tot_cells168 = n168 * len(sensor_cols)
    
    features["missingness24"] = float(h24[sensor_cols].isna().sum().sum() / tot_cells24) if tot_cells24 > 0 else np.nan
    features["missingness72"] = float(h72[sensor_cols].isna().sum().sum() / tot_cells72) if tot_cells72 > 0 else np.nan
    features["missingness168"] = float(h168[sensor_cols].isna().sum().sum() / tot_cells168) if tot_cells168 > 0 else np.nan
    
    if np.isnan(features["fail_rate24"]) or np.isnan(features["fail_rate168"]):
        features["fail_rate_trend24_vs168"] = np.nan
    else:
        features["fail_rate_trend24_vs168"] = features["fail_rate24"] - features["fail_rate168"]
        
    if n168 > 0:
        last_avail_ts = (h168["timestamp"] + delay_td).max()
        features["hours_since_last_avail"] = (origin - last_avail_ts).total_seconds() / 3600.0
    else:
        features["hours_since_last_avail"] = np.nan
        
    # Past QC EWMA with 24h half-life
    if n168 > 0:
        delays_h = (origin - (h168["timestamp"] + delay_td)).dt.total_seconds() / 3600.0
        weights = np.exp(- np.log(2) * delays_h / 24.0)
        ewma = float(np.sum(h168["fail"].values * weights) / np.sum(weights)) if np.sum(weights) > 0 else np.nan
        features["past_qc_ewma24"] = ewma
    else:
        features["past_qc_ewma24"] = np.nan

    # 8 FAST CAUSAL QC FEATURES based on AVAILABLE TIMES (timestamp + delay_h)
    avail_ts = hist["timestamp"] + delay_td
    
    # 1 & 2: qc_n_recent6 and qc_fail_rate_recent6
    rec6_mask = (avail_ts >= origin - pd.Timedelta(hours=6)) & (avail_ts <= origin)
    h_rec6 = hist.loc[rec6_mask]
    n_rec6 = len(h_rec6)
    features["qc_n_recent6"] = float(n_rec6)
    features["qc_fail_rate_recent6"] = float(h_rec6["fail"].mean()) if n_rec6 > 0 else np.nan
    
    # 3 & 4: qc_n_recent12 and qc_fail_rate_recent12
    rec12_mask = (avail_ts >= origin - pd.Timedelta(hours=12)) & (avail_ts <= origin)
    h_rec12 = hist.loc[rec12_mask]
    n_rec12 = len(h_rec12)
    features["qc_n_recent12"] = float(n_rec12)
    features["qc_fail_rate_recent12"] = float(h_rec12["fail"].mean()) if n_rec12 > 0 else np.nan
    
    # 5: qc_fail_trend6_vs24 = qc_fail_rate_recent6 - fail_rate24
    if np.isnan(features["qc_fail_rate_recent6"]) or np.isnan(features["fail_rate24"]):
        features["qc_fail_trend6_vs24"] = np.nan
    else:
        features["qc_fail_trend6_vs24"] = features["qc_fail_rate_recent6"] - features["fail_rate24"]
        
    # 6: hours_since_last_fail_available (origin - (last KNOWN Fail timestamp + delay_h))
    fails_hist = hist[hist["fail"] == 1]
    if len(fails_hist) > 0:
        last_known_fail_avail = (fails_hist["timestamp"] + delay_td).max()
        features["hours_since_last_fail_available"] = (origin - last_known_fail_avail).total_seconds() / 3600.0
    else:
        features["hours_since_last_fail_available"] = np.nan
        
    # 7 & 8: qc_last5_fail_rate and qc_last10_fail_rate
    features["qc_last5_fail_rate"] = float(hist["fail"].tail(5).mean()) if len(hist) > 0 else np.nan
    features["qc_last10_fail_rate"] = float(hist["fail"].tail(10).mean()) if len(hist) > 0 else np.nan
        
    # Calendar features at prediction time origin + lead
    target_dt = origin + pd.Timedelta(hours=lead_h)
    hour = target_dt.hour + target_dt.minute / 60.0 + target_dt.second / 3600.0
    dow = target_dt.weekday()
    
    features["sin_hour"] = np.sin(2 * np.pi * hour / 24.0)
    features["cos_hour"] = np.cos(2 * np.pi * hour / 24.0)
    features["sin_dow"] = np.sin(2 * np.pi * dow / 7.0)
    features["cos_dow"] = np.cos(2 * np.pi * dow / 7.0)
    
    # Sensor families (6 per channel)
    if n72 > 0:
        s72 = h72[sensor_cols]
        last_s = s72.ffill().iloc[-1]
        miss72 = s72.isna().mean(axis=0)
        mean72 = s72.mean(axis=0)
    else:
        last_s = pd.Series(np.nan, index=sensor_cols)
        miss72 = pd.Series(np.nan, index=sensor_cols)
        mean72 = pd.Series(np.nan, index=sensor_cols)
        
    if n24 > 0:
        s24 = h24[sensor_cols]
        mean24 = s24.mean(axis=0)
        std24 = s24.std(axis=0, ddof=1) if n24 > 1 else pd.Series(np.nan, index=sensor_cols)
        miss24 = s24.isna().mean(axis=0)
    else:
        mean24 = pd.Series(np.nan, index=sensor_cols)
        std24 = pd.Series(np.nan, index=sensor_cols)
        miss24 = pd.Series(np.nan, index=sensor_cols)
        
    delta_mean = mean24 - mean72
    
    for col in sensor_cols:
        features[f"{col}__last72"] = float(last_s[col])
        features[f"{col}__mean24"] = float(mean24[col])
        features[f"{col}__std24"] = float(std24[col])
        features[f"{col}__delta_mean24_minus_mean72"] = float(delta_mean[col])
        features[f"{col}__missing_fraction24"] = float(miss24[col])
        features[f"{col}__missing_fraction72"] = float(miss72[col])
        
    s_feat = pd.Series(features, dtype=float).replace([np.inf, -np.inf], np.nan)
    return s_feat, max_feature_available

def make_features(raw, origins=None, delay_h=6, history_h=168, lead_h=6):
    """
    Generate feature matrix across periodic grid or custom origins list.
    Default latest origin is floor(max_ts - lead_h - 6h) independent of delay_h (where min H=6h).
    """
    if origins is None:
        min_ts = raw["timestamp"].min()
        max_ts = raw["timestamp"].max()
        start_grid = (min_ts + pd.Timedelta(hours=history_h)).ceil("6h")
        end_grid = (max_ts - pd.Timedelta(hours=lead_h + 6)).floor("6h")
        origins = pd.date_range(start=start_grid, end=end_grid, freq="6h")
        
    rows = []
    meta_records = []
    for orig in origins:
        feat_s, max_avail = feature_row(raw, orig, delay_h=delay_h, history_h=history_h, lead_h=lead_h)
        rows.append(feat_s)
        meta_records.append({"origin": orig, "max_feature_available": max_avail})
        
    X = pd.DataFrame(rows, index=origins)
    meta_df = pd.DataFrame(meta_records, index=origins)
    return X, meta_df


In [4]:
# 4. Target Construction, Maturity Purging & Evaluation Metrics APIs
def build_targets(raw, origins, horizon_h, lead_h=6, delay_h=6, q=0.1, min_event_samples=5):
    """
    Construct ground-truth target windows [origin+L, origin+L+H) and maturity cutoff target_available=end+D.
    Does not filter origins based on future sample count (honest unconditioned evaluation).
    """
    records = []
    for origin in origins:
        origin = pd.Timestamp(origin)
        target_start = origin + pd.Timedelta(hours=lead_h)
        target_end = target_start + pd.Timedelta(hours=horizon_h)
        target_available = target_end + pd.Timedelta(hours=delay_h)
        
        mask = (raw["timestamp"] >= target_start) & (raw["timestamp"] < target_end)
        sub = raw.loc[mask]
        n_future = len(sub)
        n_fail = int(sub["fail"].sum()) if n_future > 0 else 0
        rate = (n_fail / n_future) if n_future > 0 else np.nan
        event = 1 if (n_future >= min_event_samples and rate >= q) else (0 if n_future >= min_event_samples else np.nan)
        
        records.append({
            "origin": origin,
            "target_start": target_start,
            "target_end": target_end,
            "target_available": target_available,
            "n_future": n_future,
            "n_fail": n_fail,
            "rate": rate,
            "event": event
        })
    return pd.DataFrame(records)

def training_maturity_gate(meta, fit_time):
    """
    Prequential maturity filter: only samples whose target outcome is fully available before fit_time (end + D <= fit_time)
    are permitted in training folds.
    """
    fit_time = pd.Timestamp(fit_time)
    return meta["target_available"] <= fit_time

def incidence(rawspan, metadata):
    """
    Compute boolean incidence matrix (M intervals x N raw rows) on half-open window [target_start, target_end).
    """
    if len(rawspan) == 0 or len(metadata) == 0:
        return np.zeros((len(metadata), len(rawspan)), dtype=bool)
    
    ts = rawspan["timestamp"].values
    starts = metadata["target_start"].values[:, None]
    ends = metadata["target_end"].values[:, None]
    return (ts >= starts) & (ts < ends)

def interval_metrics(raw, meta, warnings, eval_start, eval_end):
    """
    Evaluate primary quality-risk forecast windows with deduplicated union workload
    and honest fixed denominator. Positional-safe for DatetimeIndex and non-consecutive indices.
    """
    eval_start = pd.Timestamp(eval_start)
    eval_end = pd.Timestamp(eval_end)
    assert eval_start < eval_end, f"eval_start ({eval_start}) must be strictly < eval_end ({eval_end})"
    
    rawspan = raw[(raw["timestamp"] >= eval_start) & (raw["timestamp"] < eval_end)].copy()
    assert rawspan["raw_row_id"].is_unique, "rawspan raw_row_id must be unique"
    
    total_n = len(rawspan)
    total_fail = int(rawspan["fail"].sum())
    
    # Normalize warnings to aligned positional boolean array length of meta
    assert len(warnings) == len(meta), f"warnings length ({len(warnings)}) must equal meta length ({len(meta)})"
    if isinstance(warnings, pd.Series):
        assert warnings.index.equals(meta.index), "warnings Series index must match meta.index exactly"
        warn_bool = warnings.fillna(False).values.astype(bool)
    else:
        warn_bool = np.asarray(warnings, dtype=bool)
        
    # Target start and end validity
    if len(meta) > 0:
        assert (meta["target_end"] > meta["target_start"]).all(), "target_end must be strictly > target_start"
        leads = (meta["target_start"] - meta["origin"]).dt.total_seconds() / 3600.0
        assert (leads >= 6.0 - 1e-5).all(), "All forecast leads must be >= 6h"
        
    warn_pos = np.where(warn_bool)[0]
    warn_meta = meta.iloc[warn_pos]
    
    if len(warn_meta) > 0 and len(rawspan) > 0:
        inc = incidence(rawspan, warn_meta)
        covered_mask = inc.any(axis=0)
    else:
        covered_mask = np.zeros(len(rawspan), dtype=bool)
        
    rawspan["covered"] = covered_mask
    covered_n = int(covered_mask.sum())
    covered_fail = int(rawspan.loc[covered_mask, "fail"].sum())
    
    coverage = (covered_fail / total_fail) if total_fail > 0 else np.nan
    workload = (covered_n / total_n) if total_n > 0 else np.nan
    precision_entity = (covered_fail / covered_n) if covered_n > 0 else np.nan
    
    # Time under alert union clipped to [eval_start, eval_end)
    total_seconds_span = (eval_end - eval_start).total_seconds()
    if len(warn_meta) > 0:
        intervals = []
        for _, row in warn_meta.iterrows():
            s = max(row["target_start"], eval_start)
            e = min(row["target_end"], eval_end)
            if e > s:
                intervals.append((s, e))
        intervals.sort(key=lambda x: x[0])
        merged = []
        for s, e in intervals:
            if not merged:
                merged.append([s, e])
            else:
                if s <= merged[-1][1]:
                    merged[-1][1] = max(merged[-1][1], e)
                else:
                    merged.append([s, e])
        alert_seconds = sum((e - s).total_seconds() for s, e in merged)
        time_under_alert = alert_seconds / total_seconds_span if total_seconds_span > 0 else 0.0
    else:
        time_under_alert = 0.0
        
    # Event level evaluation on fully contained intervals
    eval_mask = (meta["target_start"] >= eval_start) & (meta["target_end"] <= eval_end)
    eval_pos = np.where(eval_mask.values)[0]
    
    if len(eval_pos) > 0:
        meta_eval = meta.iloc[eval_pos]
        warn_eval = warn_bool[eval_pos]
        
        event_vals = meta_eval["event"].values
        valid_event_mask = ~np.isnan(event_vals)
        
        if valid_event_mask.any():
            y_true = event_vals[valid_event_mask].astype(int)
            y_warn = warn_eval[valid_event_mask].astype(int)
            tp_event = int(np.sum((y_true == 1) & (y_warn == 1)))
            warned_event_total = int(np.sum(y_warn == 1))
            true_event_total = int(np.sum(y_true == 1))
            
            event_precision = (tp_event / warned_event_total) if warned_event_total > 0 else np.nan
            event_recall = (tp_event / true_event_total) if true_event_total > 0 else np.nan
            false_high_events = int(np.sum((y_true == 0) & (y_warn == 1)))
        else:
            event_precision = np.nan
            event_recall = np.nan
            false_high_events = 0
            
        sparse_high_count = int(np.sum(warn_eval & np.isnan(event_vals)))
    else:
        event_precision = np.nan
        event_recall = np.nan
        false_high_events = 0
        sparse_high_count = 0
        
    return {
        "covered_fail": covered_fail,
        "total_fail": total_fail,
        "coverage": coverage,
        "covered_n": covered_n,
        "total_n": total_n,
        "workload": workload,
        "precision_entity": precision_entity,
        "time_under_alert": time_under_alert,
        "event_precision": event_precision,
        "event_recall": event_recall,
        "sparse_high_count": sparse_high_count,
        "false_high_events": false_high_events,
        "covered_raw_ids": set(rawspan.loc[covered_mask, "raw_row_id"].values),
        "annotated_rawspan": rawspan
    }

def row_metrics(rawspan, warnings):
    """
    Evaluate secondary triage inspection metrics on individual rows.
    Coverage is row-level recall. Never confused with future forecast coverage.
    """
    assert len(rawspan) == len(warnings), "rawspan and warnings must have equal length"
    y_true = rawspan["fail"].values
    y_warn = np.array(warnings, dtype=int)
    
    tp = int(np.sum((y_true == 1) & (y_warn == 1)))
    fp = int(np.sum((y_true == 0) & (y_warn == 1)))
    fn = int(np.sum((y_true == 1) & (y_warn == 0)))
    tn = int(np.sum((y_true == 0) & (y_warn == 0)))
    
    total_n = len(rawspan)
    total_fail = int(np.sum(y_true == 1))
    warned_n = tp + fp
    
    coverage = (tp / total_fail) if total_fail > 0 else np.nan
    workload = (warned_n / total_n) if total_n > 0 else np.nan
    precision_entity = (tp / warned_n) if warned_n > 0 else np.nan
    f1 = (2 * tp / (2 * tp + fp + fn)) if (2 * tp + fp + fn) > 0 else np.nan
    
    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "covered_fail": tp,
        "total_fail": total_fail,
        "coverage": coverage,
        "covered_n": warned_n,
        "total_n": total_n,
        "workload": workload,
        "precision_entity": precision_entity,
        "F1": f1
    }


In [5]:
# 5. Online Capacity Throttling APIs: capacity_threshold & qc_threshold
def capacity_threshold(score_meta, raw, current_origin, budget=0.30, lookback_days=7, n_quantiles=21):
    """
    Select alarm threshold tau for Primary forecast using strictly matured past OOS scores.
    Guarantees no future leak: target_available <= current_origin and origin >= current_origin - lookback.
    Candidate quantiles are built ONLY from finite scores. Denominator is all past raw entities in window.
    """
    current_origin = pd.Timestamp(current_origin)
    past_start = current_origin - pd.Timedelta(days=lookback_days)
    
    m = (score_meta["target_available"] <= current_origin) & (score_meta["origin"] >= past_start)
    past_meta = score_meta.loc[m]
    
    if len(past_meta) == 0:
        return np.inf, {"max_past_availability": pd.NaT, "max_target_end": pd.NaT, "n_contexts": 0, "past_workload": 0.0}
        
    first_target_start = past_meta["target_start"].min()
    last_target_end = past_meta["target_end"].max()
    assert last_target_end <= current_origin, f"past_meta target_end ({last_target_end}) must be <= current_origin ({current_origin})"
    
    raw_hist = raw[(raw["timestamp"] >= first_target_start) & (raw["timestamp"] < last_target_end)]
    denom_n = len(raw_hist)
    
    if denom_n == 0:
        return np.inf, {"max_past_availability": past_meta["target_available"].max(), "max_target_end": last_target_end, "n_contexts": len(past_meta), "past_workload": 0.0}
        
    past_scores = past_meta["score"].values
    finite_scores = past_scores[np.isfinite(past_scores)]
    
    if len(finite_scores) == 0:
        return np.inf, {"max_past_availability": past_meta["target_available"].max(), "max_target_end": last_target_end, "n_contexts": len(past_meta), "past_workload": 0.0}
        
    qs = np.linspace(0, 1, n_quantiles)
    candidate_taus = sorted(list(set(np.quantile(finite_scores, qs)))) + [np.inf]
    
    inc_mat = incidence(raw_hist, past_meta)
    
    valid_candidates = []
    for tau in candidate_taus:
        if np.isinf(tau):
            w = 0.0
        else:
            active_m = np.isfinite(past_scores) & (past_scores >= tau)
            if active_m.any():
                cov = inc_mat[active_m].any(axis=0)
                w = cov.sum() / denom_n
            else:
                w = 0.0
        if w <= budget:
            valid_candidates.append((w, tau))
            
    if not valid_candidates:
        best_tau = np.inf
        best_workload = 0.0
    else:
        valid_candidates.sort(key=lambda x: (x[0], x[1]))
        best_workload, best_tau = valid_candidates[-1]
        
    audit = {
        "max_past_availability": past_meta["target_available"].max(),
        "max_target_end": last_target_end,
        "n_contexts": len(past_meta),
        "past_workload": float(best_workload)
    }
    return float(best_tau), audit

def qc_threshold(score_records, current_origin, budget=0.30, lookback_days=14):
    """
    Select threshold tau for Secondary QC triage using strictly historical sample scores
    whose timestamp + 6h <= current_origin.
    Candidate quantiles are built ONLY from finite scores. Denominator is all past samples.
    """
    current_origin = pd.Timestamp(current_origin)
    past_start = current_origin - pd.Timedelta(days=lookback_days)
    
    m = ((score_records["timestamp"] + pd.Timedelta(hours=6)) <= current_origin) & (score_records["timestamp"] >= past_start)
    past = score_records.loc[m]
    
    if len(past) == 0:
        return np.inf, {"max_past_availability": pd.NaT, "n_contexts": 0, "past_workload": 0.0}
        
    scores = past["score"].values
    finite_scores = scores[np.isfinite(scores)]
    
    if len(finite_scores) == 0:
        return np.inf, {"max_past_availability": (past["timestamp"] + pd.Timedelta(hours=6)).max(), "n_contexts": len(past), "past_workload": 0.0}
        
    qs = np.linspace(0, 1, 21)
    candidate_taus = sorted(list(set(np.quantile(finite_scores, qs)))) + [np.inf]
    
    denom_n = len(past)
    valid_candidates = []
    for tau in candidate_taus:
        if np.isinf(tau):
            w = 0.0
        else:
            active_m = np.isfinite(scores) & (scores >= tau)
            w = np.sum(active_m) / denom_n
        if w <= budget:
            valid_candidates.append((w, tau))
            
    if not valid_candidates:
        best_tau = np.inf
        best_workload = 0.0
    else:
        valid_candidates.sort(key=lambda x: (x[0], x[1]))
        best_workload, best_tau = valid_candidates[-1]
        
    audit = {
        "max_past_availability": (past["timestamp"] + pd.Timedelta(hours=6)).max(),
        "n_contexts": len(past),
        "past_workload": float(best_workload)
    }
    return float(best_tau), audit


In [6]:
# 6. Automated Small Synthetic Test Suite
def run_synthetic_tests():
    print("Starting synthetic integrity test suite...")
    
    # Base synthetic timeline with 590 sensor columns
    t0 = pd.Timestamp("2008-01-05 12:00:00")
    np.random.seed(42)
    n_rows = 6
    sensor_data = np.random.randn(n_rows, 590)
    sensor_cols = [f"sensor_{i}" for i in range(590)]
    
    timestamps = [
        t0 - pd.Timedelta(hours=10),                 # Row 0: avail at t0-4h <= t0, fail=0
        t0 - pd.Timedelta(hours=6),                  # Row 1: tie 1, avail at t0-6h+6h == t0, fail=0
        t0 - pd.Timedelta(hours=6),                  # Row 2: tie 2 duplicate at exact same timestamp, fail=1
        t0 - pd.Timedelta(hours=4),                  # Row 3: avail at t0+2h > t0, fail=1
        t0 + pd.Timedelta(hours=6),                  # Row 4: exactly at target_start, fail=1
        t0 + pd.Timedelta(hours=12)                  # Row 5: exactly at target_end, fail=1
    ]
    fails = [0, 0, 1, 1, 1, 1]
    
    df_syn = pd.DataFrame(sensor_data, columns=sensor_cols)
    df_syn["raw_row_id"] = np.arange(n_rows)
    df_syn["timestamp"] = timestamps
    df_syn["fail"] = fails
    
    # TEST 1: Half-open boundary on target windows [start, end)
    targets = build_targets(df_syn, origins=[t0], horizon_h=6, lead_h=6, delay_h=6, min_event_samples=5)
    assert len(targets) == 1, "Targets should have 1 record"
    assert targets.loc[0, "n_future"] == 1, f"Expected 1 future sample, got {targets.loc[0, 'n_future']}"
    assert targets.loc[0, "n_fail"] == 1, f"Expected 1 future fail, got {targets.loc[0, 'n_fail']}"
    assert np.isnan(targets.loc[0, "event"]), "Event must be NaN when n_future < min_event_samples (5)"
    print("  [PASS] Test 1: Half-open boundary correctly includes start, excludes end, and sets NaN for sparse events")
    
    # TEST 2: Delayed availability & Duplicate Ties (timestamp + delay == origin)
    feat, max_avail = feature_row(df_syn, t0, delay_h=6, history_h=168, lead_h=6)
    # Available at t0: row 0 (t0-4h), row 1 (t0), row 2 (t0). Total available = 3
    assert feat["n_avail24"] == 3.0, f"Expected n_avail24=3 (including both ties), got {feat['n_avail24']}"
    assert max_avail == t0, f"Expected max_avail == t0, got {max_avail}"
    print("  [PASS] Test 2: Delayed availability correctly preserves all simultaneous timestamp ties arriving at origin boundary")
    
    # TEST 3: Mutate all future values (across all 590 sensors) => entire feature vector incl all 8 QC features must be perfectly invariant
    df_mut = df_syn.copy()
    fut_idx = df_mut["timestamp"] > t0 - pd.Timedelta(hours=6)
    df_mut.loc[fut_idx, "fail"] = 0
    df_mut.loc[fut_idx, sensor_cols] = 9999.0
    feat_mut, max_avail_mut = feature_row(df_mut, t0, delay_h=6, history_h=168, lead_h=6)
    pd.testing.assert_series_equal(feat, feat_mut)
    assert max_avail == max_avail_mut
    print("  [PASS] Test 3: Mutating unavailable future data across all 590 sensors leaves entire causal Series identical")
    
    # TEST 3B: Fast-window QC features test for records at available time boundary and non-negative staleness
    assert feat["qc_n_recent6"] == 3.0
    assert feat["qc_fail_rate_recent6"] == (1.0 / 3.0)
    assert feat["qc_n_recent12"] == 3.0
    assert feat["qc_fail_rate_recent12"] == (1.0 / 3.0)
    assert feat["qc_fail_trend6_vs24"] == 0.0
    assert feat["hours_since_last_fail_available"] == 0.0  # Row 2 (fail=1) avail exactly at t0
    assert feat["hours_since_last_fail_available"] >= 0.0
    assert feat["hours_since_last_avail"] >= 0.0
    assert feat["qc_last5_fail_rate"] == (1.0 / 3.0)
    assert feat["qc_last10_fail_rate"] == (1.0 / 3.0)
    print("  [PASS] Test 3B: Fast-window causal QC features correctly evaluate available-time boundary and non-negative staleness")

    # TEST 4: Training maturity purge gate (target_available <= fit_time)
    t_gate_before = training_maturity_gate(targets, t0 + pd.Timedelta(hours=17))
    t_gate_after = training_maturity_gate(targets, t0 + pd.Timedelta(hours=18))
    assert not t_gate_before.iloc[0], "Sample must not be available before end + D"
    assert t_gate_after.iloc[0], "Sample must be mature and available at end + D"
    print("  [PASS] Test 4: Training maturity gate correctly purges uncompleted future windows")
    
    # TEST 5: Interval metrics with DatetimeIndex and non-contiguous indices (Positional Safety)
    dt_idx = pd.date_range("2008-01-01", periods=2, freq="12h")
    meta_dt = pd.DataFrame([
        {"origin": t0, "target_start": t0 + pd.Timedelta(hours=6), "target_end": t0 + pd.Timedelta(hours=12), "event": np.nan, "n_future": 1},
        {"origin": t0 + pd.Timedelta(hours=2), "target_start": t0 + pd.Timedelta(hours=8), "target_end": t0 + pd.Timedelta(hours=14), "event": np.nan, "n_future": 1}
    ], index=dt_idx)
    warnings_series = pd.Series([True, False], index=dt_idx)
    res_metrics = interval_metrics(df_syn, meta_dt, warnings_series, eval_start=t0+pd.Timedelta(hours=6), eval_end=t0+pd.Timedelta(hours=14))
    assert res_metrics["covered_fail"] == 1
    assert res_metrics["sparse_high_count"] == 1
    print("  [PASS] Test 5: Interval metrics handles DatetimeIndex and non-contiguous indices without positional alignment bugs")
    
    # TEST 6: Zero Fail handling -> coverage must return NaN
    df_no_fail = df_syn.copy()
    df_no_fail["fail"] = 0
    res_zero = interval_metrics(df_no_fail, meta_dt, warnings_series, eval_start=t0+pd.Timedelta(hours=6), eval_end=t0+pd.Timedelta(hours=14))
    assert np.isnan(res_zero["coverage"])
    assert res_zero["total_fail"] == 0
    print("  [PASS] Test 6: Zero Fail evaluation returns honest NaN with explicit denominator 0/0")
    
    # TEST 7: Threshold finite quantile safety & all -inf never-alert
    score_records = pd.DataFrame([
        {"origin": t0 - pd.Timedelta(days=5), "target_start": t0 - pd.Timedelta(days=5)+pd.Timedelta(hours=6), "target_end": t0 - pd.Timedelta(days=5)+pd.Timedelta(hours=12), "target_available": t0 - pd.Timedelta(days=4), "score": -np.inf},
        {"origin": t0 - pd.Timedelta(days=2), "target_start": t0 - pd.Timedelta(days=2)+pd.Timedelta(hours=6), "target_end": t0 - pd.Timedelta(days=2)+pd.Timedelta(hours=12), "target_available": t0 - pd.Timedelta(days=1), "score": -np.inf}
    ])
    raw_for_thresh = pd.DataFrame([
        {"raw_row_id": 10, "timestamp": t0 - pd.Timedelta(days=5)+pd.Timedelta(hours=8), "fail": 0},
        {"raw_row_id": 11, "timestamp": t0 - pd.Timedelta(days=2)+pd.Timedelta(hours=8), "fail": 1}
    ])
    tau_inf, audit_inf = capacity_threshold(score_records, raw_for_thresh, current_origin=t0, budget=0.5)
    assert np.isinf(tau_inf) and tau_inf > 0, f"All -inf scores must return +inf never-alert, got {tau_inf}"
    assert audit_inf["past_workload"] == 0.0
    
    # Mixture of NaN, -inf, and finite scores
    score_records_mix = pd.DataFrame([
        {"origin": t0 - pd.Timedelta(days=5), "target_start": t0 - pd.Timedelta(days=5)+pd.Timedelta(hours=6), "target_end": t0 - pd.Timedelta(days=5)+pd.Timedelta(hours=12), "target_available": t0 - pd.Timedelta(days=4), "score": np.nan},
        {"origin": t0 - pd.Timedelta(days=3), "target_start": t0 - pd.Timedelta(days=3)+pd.Timedelta(hours=6), "target_end": t0 - pd.Timedelta(days=3)+pd.Timedelta(hours=12), "target_available": t0 - pd.Timedelta(days=2), "score": -np.inf},
        {"origin": t0 - pd.Timedelta(days=2), "target_start": t0 - pd.Timedelta(days=2)+pd.Timedelta(hours=6), "target_end": t0 - pd.Timedelta(days=2)+pd.Timedelta(hours=12), "target_available": t0 - pd.Timedelta(days=1), "score": 0.75},
        {"origin": t0 + pd.Timedelta(hours=2), "target_start": t0 + pd.Timedelta(hours=8), "target_end": t0 + pd.Timedelta(hours=14), "target_available": t0 + pd.Timedelta(hours=20), "score": 0.99}
    ])
    tau_mix, audit_mix = capacity_threshold(score_records_mix, raw_for_thresh, current_origin=t0, budget=1.0)
    assert np.isfinite(tau_mix) and tau_mix == 0.75
    assert audit_mix["n_contexts"] == 3
    assert "max_target_end" in audit_mix
    print("  [PASS] Test 7: Threshold selection safely handles -inf, NaN, and finite score mixtures without future leaks")
    print("All synthetic tests completed successfully!")

if __name__ == "__main__":
    run_synthetic_tests()


Starting synthetic integrity test suite...
  [PASS] Test 1: Half-open boundary correctly includes start, excludes end, and sets NaN for sparse events
  [PASS] Test 2: Delayed availability correctly preserves all simultaneous timestamp ties arriving at origin boundary
  [PASS] Test 3: Mutating unavailable future data across all 590 sensors leaves entire causal Series identical
  [PASS] Test 3B: Fast-window causal QC features correctly evaluate available-time boundary and non-negative staleness
  [PASS] Test 4: Training maturity gate correctly purges uncompleted future windows
  [PASS] Test 5: Interval metrics handles DatetimeIndex and non-contiguous indices without positional alignment bugs
  [PASS] Test 6: Zero Fail evaluation returns honest NaN with explicit denominator 0/0
  [PASS] Test 7: Threshold selection safely handles -inf, NaN, and finite score mixtures without future leaks
All synthetic tests completed successfully!
